In [15]:
from pathlib import Path
import os
import psycopg2
import json
import pandas as pd
from sqlalchemy.engine import make_url

CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "02_process_raw_data").is_dir()),
    CURRENT_DIR,
)

# DATA_ROOT phải là thư mục gốc chứa 02_process_raw_data và 03_push_to_sql.
# Mặc định dùng PROJECT_DIR; chỉ đặt TIKI_DATA_DIR nếu dữ liệu nằm ở thư mục gốc khác.
DATA_ROOT = Path(os.environ.get("TIKI_DATA_DIR", PROJECT_DIR)).resolve()

# Ưu tiên biến môi trường; local có thể cấu hình SUPABASE_DB_URL trong file .env (đã được .gitignore).
DB_URL = os.environ.get("SUPABASE_DB_URL")
if not DB_URL and (PROJECT_DIR / ".env").exists():
    for line in (PROJECT_DIR / ".env").read_text(encoding="utf-8").splitlines():
        key, separator, value = line.partition("=")
        if separator and key.strip() == "SUPABASE_DB_URL":
            DB_URL = value.strip().strip("\"'")
            break
if not DB_URL:
    raise RuntimeError("Thiếu SUPABASE_DB_URL: cấu hình biến môi trường hoặc file .env")

CHECKPOINT_FILE = DATA_ROOT / "03_push_to_sql" / "sync_checkpoint_supabase.json"
LOG_FILE = DATA_ROOT / "03_push_to_sql" / "logs" / "notebook_incremental.log"
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Data root: {DATA_ROOT}")
print("Supabase Database Connected (URL đã được cấu hình)")
print(f"Checkpoint file: {CHECKPOINT_FILE}")


Project directory: D:\TiKi-Analysic
Data root: D:\TiKi-Analysic
Supabase Database Connected (URL đã được cấu hình)
Checkpoint file: D:\TiKi-Analysic\03_push_to_sql\sync_checkpoint_supabase.json


## 1. Xem file Checkpoint hiện tại (Những gì đã lưu)

In [16]:
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        state = json.load(f)
    print(json.dumps(state, indent=2, ensure_ascii=False))
else:
    print("Chưa có file checkpoint. Sẽ được tự động tạo khi chạy nạp.")

{
  "last_sync_time": "2026-10-24 18:22:53",
  "category": {
    "synced": true,
    "synced_at": "2026-10-21 09:10:50",
    "total_categories": 26,
    "total_sub_categories": 2946
  },
  "daily_price": {
    "synced_dates": [
      "2026-10-18",
      "2026-10-19",
      "2026-10-20",
      "2026-10-21",
      "2026-10-22",
      "2026-10-23",
      "2026-10-24"
    ]
  },
  "monthly_snapshot": {
    "dim_product_listing": [
      "2026-10"
    ],
    "product_detail": [
      "2026-10"
    ],
    "product_seller": [
      "2026-10"
    ],
    "product_category_map": [
      "2026-10"
    ]
  },
  "review": {
    "composite_keys_count": 973295,
    "key_structure": [
      "review_id",
      "product_id",
      "seller_product_id"
    ],
    "last_sync_time": "2026-10-24 18:22:53"
  }
}


## 2. Chạy tiến trình nạp gia tăng thông minh (Incremental Sync)

In [17]:
from push_to_database import push_incremental_postgres

# Chạy kiểm tra: các dữ liệu đã nạp sẽ tự động [BỎ QUA], không bị nạp trùng lên Cloud
push_incremental_postgres(PROJECT_DIR, CURRENT_DIR, DB_URL, LOG_FILE)

Kết nối đến cơ sở dữ liệu Supabase PostgreSQL...

[QUY TẮC 1] Kiểm tra dữ liệu Danh mục (Category):
  [CẢNH BÁO] Thiếu một trong hai CSV category; checkpoint vẫn để chưa đồng bộ.

[QUY TẮC 2] Kiểm tra dữ liệu Giá hằng ngày (Daily Price):

[QUY TẮC 3] Kiểm tra các bảng snapshot tháng (crawl_date: YYYY-MM):

[QUY TẮC 4] Kiểm tra dữ liệu Đánh giá (fact_review) theo BỘ 3 KHÓA KẾT HỢP:
  -> Khóa lọc kết hợp: [review_id + product_id + seller_product_id]
HOÀN THÀNH TIẾN TRÌNH NẠP THÔNG MINH LÊN SUPABASE!
Tổng số dòng mới được chèn: 0 dòng
Thời gian thực thi: 6.01 giây
Trạng thái đã lưu tại: D:\TiKi-Analysic\03_push_to_sql\push_to_sql\03_push_to_sql\sync_checkpoint_supabase.json


## 3. Thống kê tổng quan dữ liệu trong Database

In [18]:
# Chuẩn hóa URL cho psycopg2 nếu URL có suffix của SQLAlchemy, postgresql+psycopg://
parsed_url = make_url(DB_URL).set(drivername="postgresql")
conn = psycopg2.connect(parsed_url.render_as_string(hide_password=False))
tables = [
    "dim_category", "dim_sub_category", "dim_product_listing",
    "product_detail", "product_seller", "product_category_map",
    "fact_daily_price", "fact_review"
]

rows = []
for t in tables:
    df_count = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {t};", conn)
    c = df_count["total"].iloc[0]
    rows.append({"Bảng": t, "Tổng số dòng": f"{c:,}"})

display(pd.DataFrame(rows))
conn.close()

C:\Users\Admin\AppData\Local\Temp\ipykernel_9676\1778826131.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {t};", conn)
C:\Users\Admin\AppData\Local\Temp\ipykernel_9676\1778826131.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {t};", conn)
C:\Users\Admin\AppData\Local\Temp\ipykernel_9676\1778826131.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql_query(f"SELECT COUN

,Bảng,Tổng số dòng
0,dim_category,26
1,dim_sub_category,"2,946"
2,dim_product_listing,"46,935"
3,product_detail,"45,413"
4,product_seller,"53,339"
5,product_category_map,"169,831"
6,fact_daily_price,"58,701"
7,fact_review,"973,295"
